<a href="https://colab.research.google.com/github/Dewdahara/DSPL-Pre-Processing/blob/main/notebooks/02_Carparts_seg_YOLO11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Checking GPU availability
!nvidia-smi

Sun Sep 20 16:24:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU: Tesla T4


In [ ]:
# Mount to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="HflQalaBLUe911H0uOVh")
project = rf.workspace("computacional-vision").project("parts-car")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Parts-car-1 in yolov11:: 100%|██████████| 12251/12251 [00:03<00:00, 3923.05it/s]


In [ ]:
!find /content -maxdepth 3 -name "data.yaml"

/content/Parts-car-1/data.yaml


In [ ]:
!ls -lah /content/Parts-car-1/

total 32K
drwxr-xr-x 5 root root 4.0K Sep 20 16:29 .
drwxr-xr-x 1 root root 4.0K Sep 20 16:29 ..
-rw-r--r-- 1 root root  542 Sep 20 16:29 data.yaml
-rw-r--r-- 1 root root  142 Sep 20 16:29 README.dataset.txt
-rw-r--r-- 1 root root 1.3K Sep 20 16:29 README.roboflow.txt
drwxr-xr-x 4 root root 4.0K Sep 20 16:29 test
drwxr-xr-x 4 root root 4.0K Sep 20 16:29 train
drwxr-xr-x 4 root root 4.0K Sep 20 16:29 valid


In [ ]:
!cp -r /content/Parts-car-1 /content/CarParts_YOLO11

In [ ]:
!ls -lah /content/CarParts_YOLO11/

total 32K
drwxr-xr-x 5 root root 4.0K Sep 20 16:31 .
drwxr-xr-x 1 root root 4.0K Sep 20 16:31 ..
-rw-r--r-- 1 root root  542 Sep 20 16:31 data.yaml
-rw-r--r-- 1 root root  142 Sep 20 16:31 README.dataset.txt
-rw-r--r-- 1 root root 1.3K Sep 20 16:31 README.roboflow.txt
drwxr-xr-x 4 root root 4.0K Sep 20 16:31 test
drwxr-xr-x 4 root root 4.0K Sep 20 16:31 train
drwxr-xr-x 4 root root 4.0K Sep 20 16:31 valid


In [ ]:
!cat /content/CarParts_YOLO11/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 19
names: ['Capo', 'Farol Dianteiro', 'Gas_clip', 'Grade vent', 'Lanterna', 'Para-Brisa', 'Para-Choque Traseiro', 'Para-choque Dianteiro', 'Paralama Dianteiro', 'Paralama Traseiro', 'Pneu', 'Porta-Mala', 'Retrovisor', 'Roda', 'Tampa porta malas', 'Teto', 'Vidro Traseiro', 'fog_lamp', 'object']

roboflow:
  workspace: computacional-vision
  project: parts-car
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/computacional-vision/parts-car/dataset/1

In [ ]:
!find /content/CarParts_YOLO11/train/labels -type f | head -1 | xargs -I {} sh -c 'echo "--- {}"; head -1 "{}"'

--- /content/CarParts_YOLO11/train/labels/917776852_rear_jpg.rf.714bc93f9632a2d7e33cb6d37a4ed82d.txt
16 0.08474374999999999 0.3388015625 0.08655937500000001 0.3470125 0.095328125 0.35064687499999997 0.1610265625 0.3542234375 0.41350625 0.35035 0.718190625 0.338359375 0.7302265625000001 0.337728125 0.8039968749999999 0.3310546875 0.8214874999999999 0.322534375 0.8219703125000001 0.3142609375 0.8215890625 0.3069890625 0.7908859374999999 0.2445890625 0.7562031250000001 0.207803125 0.7249234375 0.191309375 0.62799375 0.18703125 0.3585875 0.1953015625 0.21429218749999998 0.2098828125 0.16436718749999998 0.2394046875 0.13684375 0.264828125 0.084678125 0.3375328125 0.08474374999999999 0.3388015625


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n-seg.pt")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
results = model.train(
    data="/content/CarParts_YOLO11/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/YOLO11_CarParts",
    name="yolo11n_seg",
    exist_ok=True,
    pretrained=True
)

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/CarParts_YOLO11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_seg, nbs=64, nms=None, opset=No

In [ ]:
!ls -lah /content/YOLO11_CarParts/yolo11n_seg/weights/

total 12M
drwxr-xr-x 2 root root 4.0K Sep 20 16:41 .
drwxr-xr-x 3 root root 4.0K Sep 20 19:35 ..
-rw-r--r-- 1 root root 5.8M Sep 20 19:34 best.pt
-rw-r--r-- 1 root root 5.8M Sep 20 19:34 last.pt


In [ ]:
!cp -r /content/YOLO11_CarParts /content/drive/MyDrive/

In [ ]:
from ultralytics import YOLO
model = YOLO(
    "/content/drive/MyDrive/YOLO11_CarParts/yolo11n_seg/weights/best.pt"
)

In [ ]:
test_results = model.val(
    data="/content/CarParts_YOLO11/data.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    project="/content/YOLO11_CarParts_Evaluation",
    name="test",
    exist_ok=True
)

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-seg summary (fused): 113 layers, 2,838,273 parameters, 0 gradients, 9.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1649.0±476.6 MB/s, size: 65.4 KB)
val: Scanning /content/CarParts_YOLO11/test/labels... 255 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 255/255 1.3Kit/s 0.2s
val: New cache created: /content/CarParts_YOLO11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.0it/s 8.0s
                   all        255       2730      0.812      0.824      0.861      0.704      0.811      0.812      0.849       0.65
                  Capo        150        150      0.951      0.933      0.972      0.858      0.959      0.938      0.976      0.865
       Farol Dianteiro        136        194      0.965      0.948      0.971       0.77      0.973      0.95

In [ ]:
print("Mask Precision:", test_results.seg.mp)
print("Mask Recall:", test_results.seg.mr)
print("Mask mAP@50:", test_results.seg.map50)
print("Mask mAP@50-95:", test_results.seg.map)

Mask Precision: 0.8110754888856744
Mask Recall: 0.8123701108760437
Mask mAP@50: 0.8493919087718209
Mask mAP@50-95: 0.6501307805952579


In [ ]:
print("Box Precision:", test_results.box.mp)
print("Box Recall:", test_results.box.mr)
print("Box mAP@50:", test_results.box.map50)
print("Box mAP@50-95:", test_results.box.map)

Box Precision: 0.8122847691918225
Box Recall: 0.823570165761365
Box mAP@50: 0.8605005134507028
Box mAP@50-95: 0.7035418281102677


In [ ]:
for i, name in enumerate(model.names):
    print(
        f"{name}: "
        f"Precision={test_results.seg.p[i]:.4f}, "
        f"Recall={test_results.seg.r[i]:.4f}, "
        f"mAP50={test_results.seg.ap50[i]:.4f}, "
        f"mAP50-95={test_results.seg.ap[i]:.4f}"
    )

0: Precision=0.9591, Recall=0.9385, mAP50=0.9756, mAP50-95=0.8647
1: Precision=0.9729, Recall=0.9536, mAP50=0.9684, mAP50-95=0.7521
2: Precision=0.8840, Recall=0.7451, mAP50=0.8504, mAP50-95=0.6342
3: Precision=0.9137, Recall=0.9487, mAP50=0.9781, mAP50-95=0.8139
4: Precision=0.7491, Recall=0.8507, mAP50=0.8621, mAP50-95=0.6515
5: Precision=0.9658, Recall=0.9835, mAP50=0.9899, mAP50-95=0.9026
6: Precision=0.9280, Recall=0.8705, mAP50=0.9344, mAP50-95=0.7373
7: Precision=0.8826, Recall=0.9124, mAP50=0.9377, mAP50-95=0.6947
8: Precision=0.6364, Recall=0.6777, mAP50=0.7396, mAP50-95=0.6397
9: Precision=0.8016, Recall=0.8048, mAP50=0.8435, mAP50-95=0.4701
10: Precision=0.8689, Recall=0.8669, mAP50=0.9018, mAP50-95=0.5791
11: Precision=0.6207, Recall=0.7370, mAP50=0.7421, mAP50-95=0.6401
12: Precision=0.9474, Recall=0.9556, mAP50=0.9821, mAP50-95=0.6576
13: Precision=0.9494, Recall=0.9708, mAP50=0.9870, mAP50-95=0.8302
14: Precision=0.8165, Recall=0.8077, mAP50=0.8523, mAP50-95=0.6412
15: P

In [ ]:
model.predict(
    source="/content/CarParts_YOLO11/test/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/YOLO11_CarParts_Evaluation",
    name="test_predictions",
    exist_ok=True
)


image 1/255 /content/CarParts_YOLO11/test/images/917516386_left_jpg.rf.4a79ea4523b719bcf60a6049d6f0e12c.jpg: 640x640 1 Capo, 2 Farol Dianteiros, 1 Gas_clip, 1 Lanterna, 1 Para-Choque Traseiro, 1 Para-choque Dianteiro, 2 Paralama Traseiros, 2 Pneus, 1 Retrovisor, 2 Rodas, 9.7ms
image 2/255 /content/CarParts_YOLO11/test/images/917560919_rear_jpg.rf.04f94712d0a11ae107d04511c76c3bf3.jpg: 640x640 3 Lanternas, 1 Para-Choque Traseiro, 1 Paralama Traseiro, 1 Tampa porta malas, 1 Teto, 1 Vidro Traseiro, 39.8ms
image 3/255 /content/CarParts_YOLO11/test/images/917560921_left_jpg.rf.123330b00d518dd4dd6e23432e792545.jpg: 640x640 1 Capo, 1 Para-Brisa, 2 Paralama Traseiros, 2 Pneus, 3 Retrovisors, 2 Rodas, 1 Teto, 24.5ms
image 4/255 /content/CarParts_YOLO11/test/images/917560954_front_jpg.rf.852e05b7e4ee5941d666e12e2eb30182.jpg: 640x640 1 Capo, 2 Farol Dianteiros, 1 Grade vent, 1 Para-Brisa, 1 Para-choque Dianteiro, 1 Retrovisor, 1 Teto, 15.4ms
image 5/255 /content/CarParts_YOLO11/test/images/917564

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'Capo', 1: 'Farol Dianteiro', 2: 'Gas_clip', 3: 'Grade vent', 4: 'Lanterna', 5: 'Para-Brisa', 6: 'Para-Choque Traseiro', 7: 'Para-choque Dianteiro', 8: 'Paralama Dianteiro', 9: 'Paralama Traseiro', 10: 'Pneu', 11: 'Porta-Mala', 12: 'Retrovisor', 13: 'Roda', 14: 'Tampa porta malas', 15: 'Teto', 16: 'Vidro Traseiro', 17: 'fog_lamp', 18: 'object'}
 obb: None
 orig_img: array([[[ 10,  10,  10],
         [ 10,  10,  10],
         [ 10,  10,  10],
         ...,
         [ 18,   5,   7],
         [ 19,   6,   8],
         [ 18,   5,   7]],
 
        [[ 11,  11,  11],
         [ 11,  11,  11],
         [ 11,  11,  11],
         ...,
         [ 18,   5,   7],
         [ 19,   6,   8],
         [ 19,   6,   8]],
 
        [[ 13,  13,  13],
         [ 13,  13,  13],
         [ 13,  13,  13],
        

In [ ]:
!cp -r /content/YOLO11_CarParts_Evaluation /content/drive/MyDrive/